In [60]:
from pymongo import MongoClient
import re

MONGO_URI = 'mongodb://18.140.62.59:27017/tc-tools'
db = MongoClient(MONGO_URI)['tc-tools']
AccountFilteredDetail = db['accounts_filtered_detail']
FollowerGroup = db["follower_group"]
VerticalKeywordGroup = db["vertical_keyword_group"]
_accounts = AccountFilteredDetail.find({"created_at": {"$gte": 1672531200}})
_account_list = FollowerGroup.find({"name_slugify" : "full-follower"})
_keywords_list = VerticalKeywordGroup.find({"name_slugify": "keyword-relevance"})

count = 0;
user_profiles = []
for i in _accounts:
    user_profiles.append(i)

print("Total: ", len(user_profiles))
for i in _account_list:
    account_list = set(i["accounts"])

for i in _keywords_list:
    keywords_list = set(i["keywords"])
    
people_keywords = ["writer", "engineer", "researcher", "manager", "professor", "speaker", "technology company", "shipping", "merch", "consult", "i am", "i'm", "memecoin", "meme coin"]
social_keywords = ["news", "trends", "talk", "podcast", "research", "course", "summit", "referral", "informative", "insight", "education", "invest", "nft collection", "nfts"]

#print("Account list", account_list)
#print("0xminion" in account_list)


('Total: ', 4834)


In [61]:
def get_alnum(string):
    if not string:
        return ''
    return ''.join(e for e in string if e.isalnum())

# Check if username is website url, or discord url in description or user url
def name_in_url(user_profile):
    username = get_alnum(user_profile["username"].replace("0x", "")).lower()
    name = get_alnum(user_profile["name"]).lower()
    if not name or not username:
        return 0

    # Check url in user url
    from_url = 1 if "user_url" in user_profile and user_profile["user_url"] and (username in user_profile["user_url"].lower() or name in user_profile["user_url"].lower()) else 0
    if from_url:
        return 1

    # Check url in description
    if user_profile["description"] and (username in user_profile["description"].lower() or name in user_profile["description"].lower()):
        # ex: a protocol for trading and automated liquidity provision on Ethereum at uniswap.org
        _data = user_profile["description"].split(username)
        for sub in _data:
            domains = re.split('; |, |\*|\n', sub)
            for domain in domains:
                if "." in domain and len(domain) >= 3:
                    return 1

        _data = user_profile["description"].split(name)
        for sub in _data:
            domains = re.split('; |, |\*|\n', sub)
            for domain in domains:
                if "." in domain and len(domain) >= 3:
                    return 1
                
    return 0


def valid_description(user_profile):
    # Check len
    valid = 1
#     if len(user_profile["description"]) < 20:
#         valid = 0
    
    # Check if description contain only link tele
    data = user_profile["description"].split(" ")
    if len(data) == 1 and "t.me" in data[0]:
        #Sample https://twitter.com/GOLDHEARTBnB
        valid = 0
    
    # Check more
    return valid

def keywords_in_description(user_profile):
    if not user_profile["description"]:
        return 0
    count = 0
    for word in keywords_list:
        if word.lower() in user_profile["description"].lower() :
            count += 1
    
    return count

def people_keywords_in_description(user_profile):
    if not user_profile["description"]:
        return 0
    
    if any(word.lower() in user_profile["description"].lower() for word in people_keywords):
        return 1

    return 0

def social_keywords_in_description(user_profile):
    if not user_profile["description"]:
        return 0
    
    if any(word.lower() in user_profile["description"].lower() for word in social_keywords):
        return 1

    return 0


# Check if user have url
def have_url(user_profile):
    return 1 if user_profile["user_url"] else 0

def is_protocol(user_profile):
    if not user_profile["description"]:
        return 0
    if not name_in_url(user_profile):
        return 0
    
    return 1

def number_friendship(user_profile):
    friendship = user_profile["friendship"]
    valid_friendship = list(set(friendship) & set(account_list))
                            
    return len(valid_friendship)

# gold = []
# blue = []
# normal = []
output = []

max_keyword_relevance = 0
max_follower_quality = 0
max_verification_status = 0
max_recency = 0

for i in user_profiles:
#     print valid_description(i)
#     print is_protocol(i) 
#     print number_friendship(i)
#     print keywords_in_description(i)
#     print people_keywords_in_description(i)
#     print social_keywords_in_description(i)
#     print "----"
    if valid_description(i) and is_protocol(i) and (number_friendship(i) > 0 or keywords_in_description(i) > 0) and not people_keywords_in_description(i) and not social_keywords_in_description(i):
        
        url = str(keywords_in_description(i)) + "," + str(number_friendship(i)) +  ",https://twitter.com/" + i["username"]
        #url = "https://twitter.com/" + i["username"]
        i["keyword_relevance"] = keywords_in_description(i)
        i["follower_quality"] = number_friendship(i)
        i["verification_status"] = 100 if i["verified_type"] == "gold" else (40 if i["verified_type"] == "blue" else 10)
        i["recency"] = i["created_at"]
        i["twitter_url"] = "https://twitter.com/" + i["username"]
        
        if i["keyword_relevance"] > max_keyword_relevance:
            max_keyword_relevance = i["keyword_relevance"]
        if i["follower_quality"] > max_follower_quality:
            max_follower_quality = i["follower_quality"]
        if i["verification_status"] > max_verification_status:
            max_verification_status = i["verification_status"]
        if i["recency"] > max_recency:
            max_recency = i["recency"]
            
        output.append(i)

print max_keyword_relevance, max_follower_quality, max_verification_status, max_recency


4 23 100 1688169600


In [62]:
def scoring(user_profiles_ext):
    for i in user_profiles_ext:
        i["score_keyword_relevance"] = i["keyword_relevance"] * 100.0 / max_keyword_relevance
        i["score_follower_quality"] = i["follower_quality"] * 100.0 / max_follower_quality
        i["score_verification_status"] = i["verification_status"] * 100.0 / max_verification_status
        i["score_recency"] = (i["recency"]/86400 - 1672444800/86400) * 100.0 / (max_recency/86400 - 1672444800/86400)
        
        i["total_score"] = i["score_keyword_relevance"]*0.25 + i["score_follower_quality"]*0.25 + i["score_verification_status"]*0.25 + i["score_recency"]*0.25

    return user_profiles_ext


sorted_user_profiles = sorted(scoring(output), key=lambda x: x['total_score'], reverse=True)
print "Total:", len(sorted_user_profiles)
print "total_score,keyword_relevance,follower_quality,verification_status,recency,url, from_last_check"
for i in sorted_user_profiles:
    print str(i["total_score"]) + "," + str(i["score_keyword_relevance"]) + "," + str(i["score_follower_quality"]) + "," + str(i["score_verification_status"]) + "," + str(i["score_recency"]) + "," + i["twitter_url"] + ","
        
    #print str(i["total_score"]) + "," + str(i["keyword_relevance"]) + "," + str(i["follower_quality"]) + "," + str(i["verification_status"]) + "," + str(i["recency"])
#     print "--------"
    #print '"' + i["username"] + '",'


Total: 158
total_score,keyword_relevance,follower_quality,verification_status,recency,url, from_last_check
64.1208791209,50.0,100.0,40.0,66.4835164835,https://twitter.com/PrismaFi,
56.8830625896,25.0,69.5652173913,100.0,32.967032967,https://twitter.com/LineaBuild,
45.7417582418,50.0,0.0,100.0,32.967032967,https://twitter.com/asymetrix_eth,
43.2417582418,100.0,0.0,40.0,32.967032967,https://twitter.com/Neutroswap,
41.25,75.0,0.0,40.0,50.0,https://twitter.com/SleekWallet,
40.0,50.0,0.0,10.0,100.0,https://twitter.com/crilenetwork,
39.3956043956,100.0,0.0,40.0,17.5824175824,https://twitter.com/JewelSwapX,
39.1208791209,50.0,0.0,40.0,66.4835164835,https://twitter.com/LineaBank,
39.1208791209,50.0,0.0,40.0,66.4835164835,https://twitter.com/MendiFinance,
39.1208791209,50.0,0.0,40.0,66.4835164835,https://twitter.com/SeiStorm,
36.9917582418,75.0,0.0,40.0,32.967032967,https://twitter.com/raft_fi,
36.9917582418,75.0,0.0,40.0,32.967032967,https://twitter.com/futuristicswap,
35.8791208791,50.0,0.0,1